In [ ]:
#installation et téléchargement
!pip install -q xlrd spacy pandas networkx nltk
!python -m spacy download es_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 27.1 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:

import os
import string
import networkx as nx
import xlrd
import pandas as pd
import spacy
import nltk

# Téléchargement des stopwords NLTK
nltk.download('stopwords', quiet=True)
nlp = spacy.load("es_core_news_sm")

#Construction des graphes NAP (Frecuencia, Tiempo, Asociación)
def Construye_Grafo():
    nombre_archivo = "NAP.xls"
    if not os.path.exists(nombre_archivo):
        raise FileNotFoundError(f"Falta el archivo {nombre_archivo} en el directorio actual.")

    libro = xlrd.open_workbook(nombre_archivo)
    hoja = libro.sheet_by_index(0)
    GrafoFrec = nx.Graph()
    GrafoTiempo = nx.Graph()
    GrafoAsoc = nx.Graph()
    num_filas = hoja.nrows
    estimulo = ""

    for fila in range(0, num_filas-1):
        celda = hoja.cell(fila, 0)
        cadena = celda.value
        if cadena == "--PALABRAS--":
            continue
        elif cadena == "======":
            estimulo = hoja.cell(fila+1, 4).value
        elif cadena == "" or cadena == "=":
            continue
        elif hoja.cell(fila, 2).value == "":
            continue
        else:
            frecuencia = hoja.cell(fila, 1).value
            tiempo = hoja.cell(fila, 2).value
            asociado = hoja.cell(fila, 3).value
            palabra = hoja.cell(fila, 4).value
            estimulo = estimulo.strip()
            lema = str(palabra).strip()

            GrafoFrec.add_edge(estimulo, lema, weight=1/float(frecuencia), sim=float(frecuencia))
            GrafoTiempo.add_edge(estimulo, lema, weight=float(tiempo), sim=1/(float(tiempo)+1))
            GrafoAsoc.add_edge(estimulo, lema, weight=100-float(asociado), sim=float(asociado))

    return GrafoFrec, GrafoTiempo, GrafoAsoc

print("Cargando grafos desde NAP.xls...")
GrafoFrec, GrafoTiempo, GrafoAsoc = Construye_Grafo()
print(f"Grafo de Asociación cargado exitosamente: {GrafoAsoc.number_of_nodes()} nodos, {GrafoAsoc.number_of_edges()} arêtes.")


# Fonctions de prétraitement et de recherche
def limpia_lematiza(cadena):
    limpiado = ""
    palabras_funcionales = nltk.corpus.stopwords.words('spanish')
    for c in string.punctuation:
        cadena = cadena.replace(c, "")
    cadena = cadena.strip()
    for palabra in cadena.split():
        if palabra not in palabras_funcionales:
            doc = nlp(palabra)
            limpiado += doc[0].lemma_ + " "
    return limpiado

def conceptos(lista, subconjunto):
    datos = []
    tope = len(lista) if len(lista) <= 100 else 100
    for x in range(0, tope):
        if str(lista[x][0]) not in subconjunto and float(lista[x][1]) > 0:
            datos.append(str(lista[x][0]))
    return datos

def reducir_grafo(subconjunto, grafo_original):
    nuevos_nodos = list(subconjunto)
    for nodo in subconjunto:
        if nodo in grafo_original:
            vecinos = grafo_original[nodo]
            for vecino in vecinos:
                nuevos_nodos.append(vecino)
    return grafo_original.subgraph(nuevos_nodos)

# Modèle PageRank personnalisé (the best quoi)
def diccionario_pr(grafo, definicion, alpha=0.5):
    texto = limpia_lematiza(definicion).lower()
    tokens = texto.split(" ")
    subconjunto_lemas = [token for token in tokens if token in grafo.nodes() and token != ""]

    if len(subconjunto_lemas) > 0:
        pers = {n: (1 if n in subconjunto_lemas else 0) for n in grafo.nodes()}
        resultado = nx.pagerank(grafo, alpha=alpha, personalization=pers, weight="sim")
        encontrado = sorted(resultado.items(), key=lambda x: x[1], reverse=True)[0:99]
        return conceptos(encontrado, subconjunto_lemas)
    else:
        return []

#Grapho réduit ie cercania
def diccionario_cc(grafo, definicion):
    texto = limpia_lematiza(definicion).lower()
    tokens = texto.split(" ")
    subconjunto_lemas = [token for token in tokens if token in grafo.nodes() and token != ""]

    if len(subconjunto_lemas) > 0:
        grafo_reducido = reducir_grafo(subconjunto_lemas, grafo)
        resultado = nx.closeness_centrality(grafo_reducido, distance="weight")
        encontrado = sorted(resultado.items(), key=lambda x: x[1], reverse=True)[0:99]
        return conceptos(encontrado, subconjunto_lemas)
    else:
        return []


#CELLULE DE DÉMONSTRATION INTERACTIVE
def demo_interactiva():
    print("\n" + "="*50)
    print("DEMO EN DIRECT - DICCIONARIO INVERSO (NAP)")
    print("Modèle actif : PageRank Personalizado (alpha=0.5)")
    print("Escribe una descripción (ej: 'animal que ladra', 'aparato para limpiar polvo')")
    print("Escribe 'exit' o deja vacío para salir.")
    print("="*50 + "\n")

    while True:
        definicion = input("Introduce tu descripción: ").strip()
        if not definicion or definicion.lower() == 'exit':
            print("¡Gracias por la demostración! Suerte en la presentación")
            break

        # Exécution du modèle PageRank
        candidatos = diccionario_pr(GrafoAsoc, definicion, alpha=0.5)[:5]

        print(f"    Definición: « {definicion} »")
        if candidatos:
            print(f"    Top 5 Candidatos: {candidatos}")
            print(f"    Mejor opción propuesta: {candidatos[0]}")
        else:
            print("    No se encontraron candidatos (prueba con otras palabras o verifica los lemas).")
        print("-" * 50)


demo_interactiva()

Cargando grafos desde NAP.xls...
Grafo de Asociación cargado exitosamente: 4239 nodos, 12872 arêtes.

DEMO EN DIRECT - DICCIONARIO INVERSO (NAP)
Modèle actif : PageRank Personalizado (alpha=0.5)
Escribe una descripción (ej: 'animal que ladra', 'aparato para limpiar polvo')
Escribe 'exit' o deja vacío para salir.

    Definición: « objeto que sirve para ver la hora »
    Top 5 Candidatos: ['reloj', 'plato', 'ojo', 'lente', 'tiempo']
    Mejor opción propuesta: reloj
--------------------------------------------------
